In [0]:
from pyspark.sql import functions as F

order_items_bronze = spark.table(
    "workspace.fashion_bronze.order_items_ingested"
)

print("Bronze order item records:", order_items_bronze.count())
order_items_bronze.printSchema()

Bronze order item records: 499765
root
 |-- product_id: integer (nullable = true)
 |-- order_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- gross_amount: double (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- discount_amount: double (nullable = true)
 |-- net_amount: double (nullable = true)



In [0]:
order_items_bronze.agg(
    F.count("*").alias("total_records"),
    F.countDistinct("order_id").alias("unique_orders"),
    F.countDistinct("product_id").alias("unique_products"),
    F.sum("quantity").alias("total_units_sold"),
    F.min("quantity").alias("minimum_quantity"),
    F.max("quantity").alias("maximum_quantity"),
    F.round(F.sum("gross_amount"), 2).alias("total_gross_amount"),
    F.round(F.sum("discount_amount"), 2).alias("total_discount"),
    F.round(F.sum("net_amount"), 2).alias("total_net_revenue"),
    F.round(F.avg("net_amount"), 2).alias("average_line_item_value")
).show(truncate=False)

+-------------+-------------+---------------+----------------+----------------+----------------+------------------+--------------+-----------------+-----------------------+
|total_records|unique_orders|unique_products|total_units_sold|minimum_quantity|maximum_quantity|total_gross_amount|total_discount|total_net_revenue|average_line_item_value|
+-------------+-------------+---------------+----------------+----------------+----------------+------------------+--------------+-----------------+-----------------------+
|499765       |200000       |10000          |998379          |1               |3               |3.12737749783E9   |1.9207661913E8|2.9353008787E9   |5873.36                |
+-------------+-------------+---------------+----------------+----------------+----------------+------------------+--------------+-----------------+-----------------------+



In [0]:
order_items_bronze.groupBy("category") \
    .agg(
        F.count("*").alias("line_items"),
        F.sum("quantity").alias("units_sold"),
        F.round(F.sum("net_amount"), 2).alias("net_revenue")
    ) \
    .orderBy(F.desc("net_revenue")) \
    .show(truncate=False)

order_items_bronze.groupBy("discount_percent") \
    .count() \
    .orderBy("discount_percent") \
    .show()

+----------------+----------+----------+--------------+
|category        |line_items|units_sold|net_revenue   |
+----------------+----------+----------+--------------+
|Ethnic Wear     |71722     |142954    |7.280145401E8 |
|Footwear        |72026     |144173    |5.1661380414E8|
|Women's Clothing|71917     |143487    |4.3211899707E8|
|Sportswear      |71779     |143337    |3.6970379464E8|
|Accessories     |72222     |144301    |3.5279025328E8|
|Men's Clothing  |68315     |136539    |3.4595964508E8|
|Kids' Clothing  |71784     |143588    |1.9009984439E8|
+----------------+----------+----------+--------------+

+----------------+------+
|discount_percent| count|
+----------------+------+
|               0|200437|
|               5|124632|
|              10| 75008|
|              15| 60276|
|              20| 39412|
+----------------+------+



In [0]:
print("Total records:", order_items_bronze.count())

print("Duplicate order-product combinations:",
      order_items_bronze
      .groupBy("order_id", "product_id")
      .count()
      .filter(F.col("count") > 1)
      .count())

for column in order_items_bronze.columns:
    null_count = order_items_bronze.filter(
        F.col(column).isNull()
    ).count()
    print(f"{column}: {null_count} null values")

Total records: 499765
Duplicate order-product combinations: 52
product_id: 0 null values
order_id: 0 null values
quantity: 0 null values
category: 0 null values
subcategory: 0 null values
brand: 0 null values
price: 0 null values
gross_amount: 0 null values
discount_percent: 0 null values
discount_amount: 0 null values
net_amount: 0 null values


In [0]:
duplicate_combinations = (
    order_items_bronze
    .groupBy("order_id", "product_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate combinations:", duplicate_combinations.count())

order_items_bronze.join(
    duplicate_combinations.select("order_id", "product_id"),
    on=["order_id", "product_id"],
    how="inner"
).orderBy("order_id", "product_id").show(20, truncate=False)

Duplicate combinations: 52
+-----------+----------+--------+----------------+-------------+-------------+-------+------------+----------------+---------------+----------+
|order_id   |product_id|quantity|category        |subcategory  |brand        |price  |gross_amount|discount_percent|discount_amount|net_amount|
+-----------+----------+--------+----------------+-------------+-------------+-------+------------+----------------+---------------+----------+
|ORD00000974|7550      |1       |Accessories     |Backpacks    |House of Nova|1727.25|1727.25     |5               |86.36          |1640.89   |
|ORD00000974|7550      |3       |Accessories     |Backpacks    |House of Nova|1727.25|5181.75     |5               |259.09         |4922.66   |
|ORD00008072|6072      |3       |Women's Clothing|Jeans        |Aura Studio  |2821.04|8463.12     |0               |0.0            |8463.12   |
|ORD00008072|6072      |2       |Women's Clothing|Jeans        |Aura Studio  |2821.04|5642.08     |15        

In [0]:
order_items_bronze.agg(
    F.sum(F.when(F.col("quantity") <= 0, 1).otherwise(0)).alias("invalid_quantities"),
    F.sum(F.when(F.col("price") <= 0, 1).otherwise(0)).alias("invalid_prices"),
    F.sum(F.when(~F.col("discount_percent").isin(0, 5, 10, 15, 20), 1).otherwise(0)).alias("invalid_discount_percentages"),
    F.sum(F.when(F.abs(F.col("gross_amount") - F.col("price") * F.col("quantity")) > 0.01, 1).otherwise(0)).alias("gross_amount_mismatches"),
    F.sum(F.when(F.abs(F.col("discount_amount") - F.col("gross_amount") * F.col("discount_percent") / 100) > 0.01, 1).otherwise(0)).alias("discount_mismatches"),
    F.sum(F.when(F.abs(F.col("net_amount") - (F.col("gross_amount") - F.col("discount_amount"))) > 0.01, 1).otherwise(0)).alias("net_amount_mismatches")
).show(truncate=False)

+------------------+--------------+----------------------------+-----------------------+-------------------+---------------------+
|invalid_quantities|invalid_prices|invalid_discount_percentages|gross_amount_mismatches|discount_mismatches|net_amount_mismatches|
+------------------+--------------+----------------------------+-----------------------+-------------------+---------------------+
|0                 |0             |0                           |0                      |0                  |0                    |
+------------------+--------------+----------------------------+-----------------------+-------------------+---------------------+



In [0]:
orders_silver = spark.table("workspace.fashion_silver.orders")
products_silver = spark.table("workspace.fashion_silver.products")

invalid_order_references = order_items_bronze.join(
    orders_silver.select("order_id"),
    on="order_id",
    how="left_anti"
)

invalid_product_references = order_items_bronze.join(
    products_silver.select("product_id"),
    on="product_id",
    how="left_anti"
)

print("Invalid order references:", invalid_order_references.count())
print("Invalid product references:", invalid_product_references.count())

Invalid order references: 0
Invalid product references: 0


In [0]:
order_items_silver = (
    order_items_bronze
    .withColumn("order_id", F.upper(F.trim(F.col("order_id"))))
    .withColumn("category", F.trim(F.col("category")))
    .withColumn("subcategory", F.trim(F.col("subcategory")))
    .withColumn("brand", F.trim(F.col("brand")))
)

order_items_silver.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- order_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- gross_amount: double (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- discount_amount: double (nullable = true)
 |-- net_amount: double (nullable = true)



In [0]:
print("Total records:", order_items_silver.count())

print("Duplicate order-product combinations:",
      order_items_silver
      .groupBy("order_id", "product_id")
      .count()
      .filter(F.col("count") > 1)
      .count())

print("Null order IDs:",
      order_items_silver.filter(F.col("order_id").isNull()).count())

print("Null product IDs:",
      order_items_silver.filter(F.col("product_id").isNull()).count())

print("Blank brand values:",
      order_items_silver.filter(
          F.trim(F.col("brand")) == ""
      ).count())

Total records: 499765
Duplicate order-product combinations: 52
Null order IDs: 0
Null product IDs: 0
Blank brand values: 0


In [0]:
order_items_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.fashion_silver.order_items")

In [0]:
saved_order_items = spark.table(
    "workspace.fashion_silver.order_items"
)

print("Saved records:", saved_order_items.count())
saved_order_items.printSchema()
saved_order_items.show(5, truncate=False)

Saved records: 499765
root
 |-- product_id: integer (nullable = true)
 |-- order_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- gross_amount: double (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- discount_amount: double (nullable = true)
 |-- net_amount: double (nullable = true)

+----------+-----------+--------+----------------+-----------+------------+-------+------------+----------------+---------------+----------+
|product_id|order_id   |quantity|category        |subcategory|brand       |price  |gross_amount|discount_percent|discount_amount|net_amount|
+----------+-----------+--------+----------------+-----------+------------+-------+------------+----------------+---------------+----------+
|238       |ORD00050000|2       |Women's Clothing|Trousers   |Everyday Co.|2739.94|547